In [48]:
!pip install torch torch-geometric networkx


In [76]:
!pip install astor

In [126]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader
import ast
from collections import defaultdict
import numpy as np
import random
from sklearn.preprocessing import LabelEncoder

import os
import pandas as pd
from ast import parse, AST
import astor
import networkx as nx
import torch
from torch_geometric.data import Data

In [81]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 28.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2


In [82]:
from datasets import load_dataset
dataset = load_dataset("code_search_net", "python")


/home/jovyan/.mlspace/envs/cdr-py310-pt251/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The repository for code_search_net contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/code_search_net.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Generating validation split: 100%|██████████| 23107/23107 [00:02<00:00, 8045.18 examples/s]


In [50]:
# def normalize_variables(node, var_counter=None, var_map=None):
#     if var_counter is None:
#         var_counter = [0]
#     if var_map is None:
#         var_map = {}
    
#     if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
#         if node.id not in var_map:
#             var_counter[0] += 1
#             var_map[node.id] = f"var{var_counter[0]}"
#         node.id = var_map[node.id]
    
#     for field, old_value in ast.iter_fields(node):
#         if isinstance(old_value, list):
#             for item in old_value:
#                 if isinstance(item, ast.AST):
#                     normalize_variables(item, var_counter, var_map)
#         elif isinstance(old_value, ast.AST):
#             normalize_variables(old_value, var_counter, var_map)
    
#     return node

# def ast_to_graph(node, parent=None, graph=None, node_counter=None, node_features=None, 
#                  function_name=None, current_function=None):
#     if graph is None:
#         graph = nx.DiGraph()
#         node_counter = [0]
#         node_features = {}
#         current_function = function_name
    
#     current_id = node_counter[0]
#     node_type = type(node).__name__
    
#     # Сохраняем информацию о типе узла и функции, к которой он принадлежит
#     node_features[current_id] = {
#         'type': node_type,
#         'function': current_function,
#         'is_function_node': isinstance(node, ast.FunctionDef)
#     }
    
#     graph.add_node(current_id, type=node_type)
    
#     if parent is not None:
#         graph.add_edge(parent, current_id)
    
#     node_counter[0] += 1
    
#     # Если это новая функция, обновляем current_function
#     if isinstance(node, ast.FunctionDef):
#         current_function = node.name
    
#     for field, value in ast.iter_fields(node):
#         if isinstance(value, ast.AST):
#             ast_to_graph(value, current_id, graph, node_counter, node_features, 
#                         function_name, current_function)
#         elif isinstance(value, list):
#             for item in value:
#                 if isinstance(item, ast.AST):
#                     ast_to_graph(item, current_id, graph, node_counter, node_features,
#                                function_name, current_function)
    
#     return graph, node_features

# def code_to_graph(code, function_name):
#     try:
#         tree = ast.parse(code)
#         normalized_tree = normalize_variables(tree)
#         graph, node_features = ast_to_graph(normalized_tree, function_name=function_name)
#         return graph, node_features
#     except Exception as e:
#         print(f"Error parsing code: {e}")
#         return None, None

# # 2. Создание датасета с информацией о функциях
# def create_dataset_from_file(filename):
#     with open(filename, 'r') as f:
#         code = f.read()
    
#     dataset = []
#     tree = ast.parse(code)
    
#     # Сначала собираем все имена функций для создания энкодера
#     function_names = []
#     for node in ast.walk(tree):
#         if isinstance(node, ast.FunctionDef):
#             function_names.append(node.name)
    
#     # Теперь обрабатываем каждую функцию
#     for node in ast.walk(tree):
#         if isinstance(node, ast.FunctionDef):
#             func_code = ast.unparse(node)
#             graph, node_features = code_to_graph(func_code, node.name)
#             if graph is not None:
#                 dataset.append((node.name, func_code, graph, node_features))
    
#     return dataset, function_names



In [51]:
!pip install chardet

In [53]:

# dataset, function_names = create_dataset_from_file("mini_code_dataset.py")

In [112]:
import ast
import networkx as nx
from datasets import load_dataset
from collections import defaultdict
from tqdm import tqdm

def normalize_variables(node, var_counter=None, var_map=None):
    """Normalize variable names to var1, var2, etc."""
    if var_counter is None:
        var_counter = [0]
    if var_map is None:
        var_map = {}
    
    if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
        if node.id not in var_map:
            var_counter[0] += 1
            var_map[node.id] = f"var{var_counter[0]}"
        node.id = var_map[node.id]
    
    for field, old_value in ast.iter_fields(node):
        if isinstance(old_value, list):
            for item in old_value:
                if isinstance(item, ast.AST):
                    normalize_variables(item, var_counter, var_map)
        elif isinstance(old_value, ast.AST):
            normalize_variables(old_value, var_counter, var_map)
    
    return node

def ast_to_graph(node, parent=None, graph=None, node_counter=None, 
                node_features=None, current_function=None):
    """Convert AST to graph representation with node features"""
    if graph is None:
        graph = nx.DiGraph()
        node_counter = [0]
        node_features = {}
        current_function = "global"
    
    current_id = node_counter[0]
    node_type = type(node).__name__
    
    node_features[current_id] = {
        'type': node_type,
        'function': current_function,
        'is_function': int(isinstance(node, ast.FunctionDef)),
        'is_control_flow': int(isinstance(node, (ast.If, ast.For, ast.While, ast.Try)))
    }
    
    graph.add_node(current_id)
    
    if parent is not None:
        graph.add_edge(parent, current_id)
    
    node_counter[0] += 1
    
    # Update current function context
    if isinstance(node, ast.FunctionDef):
        current_function = node.name

    for field, value in ast.iter_fields(node):
        if isinstance(value, ast.AST):
            ast_to_graph(value, current_id, graph, node_counter, 
                        node_features, current_function)
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, ast.AST):
                    ast_to_graph(item, current_id, graph, node_counter,
                               node_features, current_function)
    
    return graph, node_features

def code_to_graph(code_snippet, max_components=1000):
    """Convert Python code snippet to graph representation with size limit"""
    try:
        tree = ast.parse(code_snippet)
        normalized_tree = normalize_variables(tree)
        graph, node_features = ast_to_graph(normalized_tree)
        
        if len(graph.nodes) > max_components:
            nodes_to_keep = list(graph.nodes)[:max_components]
            subgraph = graph.subgraph(nodes_to_keep)
            
            # Filter node features
            filtered_features = {
                n_id: feat for n_id, feat in node_features.items() 
                if n_id in nodes_to_keep
            }
            
            return {
                'graph': subgraph,
                'node_features': filtered_features,
                'code': f"// Trimmed from {len(graph.nodes)} to {max_components} components\n{code_snippet[:500]}...",
                'num_nodes': len(subgraph.nodes),
                'num_edges': len(subgraph.edges),
                'was_trimmed': True
            }
        else:
            return {
                'graph': graph,
                'node_features': node_features,
                'code': code_snippet,
                'num_nodes': len(graph.nodes),
                'num_edges': len(graph.edges),
                'was_trimmed': False
            }
            
    except (SyntaxError, TypeError, ValueError) as e:
        return None

def create_graph_dataset(dataset, sample_size=1000):
    """Create graph dataset from CodeSearchNet"""
    graph_data = []
    type_counter = defaultdict(int)
    skipped = 0
    
    for example in tqdm(dataset.select(range(sample_size)), desc="Processing"):

        code = example['whole_func_string'] if 'whole_func_string' in example else example['func_code']
        result = code_to_graph(code)
        if result:
            graph_data.append(result)

            for features in result['node_features'].values():
                type_counter[features['type']] += 1
        else:
            skipped += 1
    
    print(f"\nProcessed {len(graph_data)} functions, skipped {skipped}")
    print(f"Node type distribution: {dict(sorted(type_counter.items(), key=lambda x: -x[1])[:10])}...")
    return graph_data

dataset = dataset = load_dataset("code_search_net", "python", split = "train")

graph_dataset = create_graph_dataset(dataset, sample_size=20000)

Processing: 100%|██████████| 5000/5000 [00:09<00:00, 552.99it/s] 


Processed 4987 functions, skipped 13
Node type distribution: {'Load': 144557, 'Name': 118819, 'Constant': 43214, 'Attribute': 36993, 'Call': 33804, 'Store': 27170, 'Assign': 19717, 'arg': 14078, 'Expr': 12235, 'Subscript': 9783}...


In [113]:
#dataset[:2]

In [114]:
from tqdm import tqdm

In [115]:



def prepare_data_with_embeddings(graph_dataset):
    data_list = []

    all_node_types = set()
    all_function_names = set()
    
    for item in graph_dataset:
        for features in item['node_features'].values():
            all_node_types.add(features['type'])
            all_function_names.add(features['function'])

    node_type_encoder = LabelEncoder().fit(list(all_node_types))
    function_encoder = LabelEncoder().fit(list(all_function_names))

    for item in tqdm(graph_dataset):
        graph = item['graph']
        node_features = item['node_features']

        edges = list(graph.edges())
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

        num_nodes = len(node_features)
        x = torch.zeros((num_nodes, 2), dtype=torch.long)
        
        for node_id, features in node_features.items():
            x[node_id, 0] = node_type_encoder.transform([features['type']])[0]
            x[node_id, 1] = function_encoder.transform([features['function']])[0]

        y = torch.full((num_nodes,), -1, dtype=torch.long)
        for i in range(num_nodes - 1):
            y[i] = i + 1

        data = Data(
            x=x,
            edge_index=edge_index,
            y=y,
            code=item['code'],
            num_nodes=num_nodes,
            function_name=features['function']
        )
        data_list.append(data)
    
    return data_list, node_type_encoder, function_encoder


data_list, node_type_encoder, function_encoder = prepare_data_with_embeddings(graph_dataset)

100%|██████████| 4987/4987 [44:20<00:00,  1.87it/s]  


In [116]:
#data_list[:5]

In [117]:
# Разделяем данные
random.shuffle(data_list)
split_idx = int(0.8 * len(data_list))
train_data = data_list[:split_idx]
val_data = data_list[split_idx:]

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64)

In [118]:
#train_data[:5]

In [119]:
device = "cpu"

In [120]:
len(train_data)

3989

In [127]:
class EnhancedGNNEncoder(nn.Module):
    def __init__(self, num_node_types, num_functions, hidden_dim=64):
        super().__init__()
        
        # Эмбеддинги для типов узлов и функций
        self.node_type_embedding = nn.Embedding(num_node_types, hidden_dim)
        #self.function_embedding = nn.Embedding(num_functions, hidden_dim // 2)
        
        # GCN слои
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        
        # Дополнительные слои для next token prediction
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        
        # Функциональный энкодер (для всей функции)
        self.function_encoder = nn.Linear(hidden_dim, hidden_dim)
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        
        # Разделяем фичи
        node_types = x[:, 0]
        function_ids = x[:, 1]
        
        # Применяем эмбеддинги
        node_type_emb = self.node_type_embedding(node_types)
        #function_emb = self.function_embedding(function_ids)
        
        # Объединяем эмбеддинги
        #x = torch.cat([node_type_emb, function_emb], dim=-1)
        x = node_type_emb
        
        # GCN слои
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        
        # Энкодирование всей функции (берем эмбеддинг узла функции)
        function_nodes = (data.x[:, 1] == function_ids[0]).nonzero().squeeze()
        function_embedding = x[function_nodes].mean(dim=0)
        function_embedding = self.function_encoder(function_embedding)
        
        # Next token prediction
        out = self.fc1(x)
        out = F.relu(out)
        out = self.fc2(out + function_embedding.unsqueeze(0).expand(out.size(0), -1))
        
        return out, function_embedding

# Инициализация модели
num_node_types = len(node_type_encoder.classes_)
num_functions = len(function_encoder.classes_)

model = EnhancedGNNEncoder(num_node_types, num_functions, hidden_dim=1024).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
criterion = nn.CrossEntropyLoss(ignore_index=-1)

In [128]:
from tqdm import tqdm

In [129]:
def train():
    model.train()
    total_loss = 0
    
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        
        out, _ = model(data)
        #print(data.y)
        loss = criterion(out, data.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def validate():
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out, _ = model(data)
            loss = criterion(out, data.y)
            total_loss += loss.item()
    
    return total_loss / len(val_loader)



In [131]:
# Тренировка
val_min = 100000
for epoch in range(1, 101):
    train_loss = train()
    val_loss = validate()
    if val_loss < val_min:
        torch.save(model, "graph_model2.pth")
        val_min = val_loss
    if epoch % 5 == 0:
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')


Epoch: 005, Train Loss: 5.4733, Val Loss: 5.5850
Epoch: 010, Train Loss: 5.4257, Val Loss: 5.5317
Epoch: 015, Train Loss: 5.4092, Val Loss: 5.5167
Epoch: 020, Train Loss: 5.3943, Val Loss: 5.5118
Epoch: 025, Train Loss: 5.3826, Val Loss: 5.5085
Epoch: 030, Train Loss: 5.3882, Val Loss: 5.5042
Epoch: 035, Train Loss: 5.3660, Val Loss: 5.4962
Epoch: 040, Train Loss: 5.3662, Val Loss: 5.4998
Epoch: 045, Train Loss: 5.3657, Val Loss: 5.4964
Epoch: 050, Train Loss: 5.3665, Val Loss: 5.4955
Epoch: 055, Train Loss: 5.3520, Val Loss: 5.4896
Epoch: 060, Train Loss: 5.3552, Val Loss: 5.4906
Epoch: 065, Train Loss: 5.3565, Val Loss: 5.4864
Epoch: 070, Train Loss: 5.3530, Val Loss: 5.4877
Epoch: 075, Train Loss: 5.3469, Val Loss: 5.4851
Epoch: 080, Train Loss: 5.3395, Val Loss: 5.4932
Epoch: 085, Train Loss: 5.3403, Val Loss: 5.4849
Epoch: 090, Train Loss: 5.3288, Val Loss: 5.4831
Epoch: 095, Train Loss: 5.3451, Val Loss: 5.4903
Epoch: 100, Train Loss: 5.3428, Val Loss: 5.4815


In [125]:
def encode_function(model, ast_graph):
    model.eval()
    data = ast_graph.to(device)
    with torch.no_grad():
        out, function_embedding = model(data)
    return out

# Пример: кодирование функции
sample_data = train_data[0].to(device)
function_embedding = encode_function(model, sample_data)
print(f"Function '{sample_data.function_name}' embedding:", function_embedding[:])  # первые 5 значений

Function 'extract_transformers_from_source' embedding: tensor([[ -5.1098,  16.0535,   7.6940,  ...,  -5.5121,  -4.9670,  -5.5997],
        [ -5.2480,   7.1573,  15.8777,  ...,  -5.8457,  -5.6462,  -5.2892],
        [ -6.6555, -12.2817,   4.5793,  ...,  -7.3614,  -7.3068,  -6.8929],
        ...,
        [ -8.8030,  -6.6681,  -7.3802,  ...,  -8.5016,  -8.6908,  -8.9005],
        [-10.6022, -12.5275, -10.8291,  ..., -10.5689, -10.7742, -10.8489],
        [ -8.1489,  -6.8168,  -6.1187,  ...,  -8.1400,  -8.2474,  -8.3308]])
